# 採用管理・ファネル分析

架空の採用管理データを使い、採用ファネル・転換率・リードタイム・媒体効果・パイプライン状況を分析します。

---

**実行環境:** MySQL 8.0 / Python 3 / pandas  
**DB:** `sql_portfolio`（`SETUP.md` の手順で事前に構築）

## セットアップ

In [1]:
import mysql.connector
import pandas as pd
from IPython.display import display, HTML

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

con = mysql.connector.connect(
    host='localhost', user='root', password='',
    database='sql_portfolio',
    charset='utf8mb4'
)

def run(sql):
    return pd.read_sql(sql, con)


---

## 分析クエリ

### 01. 求人別ファネル件数

**ビジネス課題:** 求人ごとの応募〜採用の到達人数を一覧化し、選考進捗を俯瞰する。

**使用テクニック:** `COUNT(DISTINCT CASE WHEN ...)`, `LEFT JOIN`

In [2]:
sql = '''
-- LEFT JOIN で全応募を保持しつつ、CASE式で各ステージの到達者を個別カウント
SELECT
    j.job_id,
    j.job_title,
    COUNT(DISTINCT a.application_id) AS applied_count,
    COUNT(DISTINCT CASE WHEN s.stage_name = 'interview' THEN s.application_id END) AS interview_count,
    COUNT(DISTINCT CASE WHEN s.stage_name = 'offer'     THEN s.application_id END) AS offer_count,
    COUNT(DISTINCT CASE WHEN s.stage_name = 'hired'     THEN s.application_id END) AS hired_count
FROM jobs j
JOIN applications a
  ON j.job_id = a.job_id
LEFT JOIN application_stages s
  ON a.application_id = s.application_id
GROUP BY j.job_id, j.job_title
ORDER BY applied_count DESC;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_4092\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,job_id,job_title,applied_count,interview_count,offer_count,hired_count
0,6,マーケティング,9,2,1,0
1,2,カスタマーサポート,4,2,1,1
2,4,データアナリスト,4,2,1,0
3,1,内勤営業,3,2,1,0
4,5,人事アシスタント,3,1,0,0
5,3,システムエンジニア,2,2,0,0


### 02. ファネルコンバージョン率

**ビジネス課題:** 各ステージ間の通過率を算出し、ボトルネックを特定する。

**使用テクニック:** CTE, `NULLIF`, 比率計算

In [3]:
sql = '''
-- Step 1: 求人ごとの各ステージ到達者数を集計
WITH funnel AS (
    SELECT
        j.job_id,
        j.job_title,
        COUNT(DISTINCT a.application_id) AS applied_count,
        COUNT(DISTINCT CASE WHEN s.stage_name = 'interview' THEN s.application_id END) AS interview_count,
        COUNT(DISTINCT CASE WHEN s.stage_name = 'offer'     THEN s.application_id END) AS offer_count,
        COUNT(DISTINCT CASE WHEN s.stage_name = 'hired'     THEN s.application_id END) AS hired_count
    FROM jobs j
    JOIN applications a ON j.job_id = a.job_id
    LEFT JOIN application_stages s ON a.application_id = s.application_id
    GROUP BY j.job_id, j.job_title
)
-- Step 2: ステージ間の転換率を算出（NULLIF でゼロ除算を防止）
SELECT
    job_id,
    job_title,
    applied_count,
    interview_count,
    offer_count,
    hired_count,
    ROUND(interview_count / NULLIF(applied_count, 0) * 100, 1) AS app_to_interview_pct,
    ROUND(offer_count     / NULLIF(interview_count, 0) * 100, 1) AS interview_to_offer_pct,
    ROUND(hired_count     / NULLIF(offer_count, 0) * 100, 1) AS offer_to_hire_pct,
    ROUND(hired_count     / NULLIF(applied_count, 0) * 100, 2) AS app_to_hire_pct
FROM funnel
ORDER BY app_to_hire_pct DESC;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_4092\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,job_id,job_title,applied_count,interview_count,offer_count,hired_count,app_to_interview_pct,interview_to_offer_pct,offer_to_hire_pct,app_to_hire_pct
0,2,カスタマーサポート,4,2,1,1,50.00,50.00,100.00,25.00
1,1,内勤営業,3,2,1,0,66.70,50.00,0.00,0.00
2,3,システムエンジニア,2,2,0,0,100.00,0.00,NaN,0.00
3,4,データアナリスト,4,2,1,0,50.00,50.00,0.00,0.00
4,5,人事アシスタント,3,1,0,0,33.30,0.00,NaN,0.00
5,6,マーケティング,9,2,1,0,22.20,50.00,0.00,0.00


### 03. 採用リードタイム（Time-to-Hire）

**ビジネス課題:** 応募から採用決定までの所要日数を求人別に集計する。

**使用テクニック:** `DATEDIFF`, `AVG/MIN/MAX`

In [4]:
sql = '''
-- 採用済み（hired_flag = 1）の応募に絞り、
-- 応募日〜採用日の差分を求人別に集計
SELECT
    j.job_id,
    j.job_title,
    COUNT(*) AS hired_count,
    ROUND(AVG(DATEDIFF(a.hired_at, a.applied_at)), 1) AS avg_time_to_hire_days,
    MIN(DATEDIFF(a.hired_at, a.applied_at))           AS min_time_to_hire_days,
    MAX(DATEDIFF(a.hired_at, a.applied_at))           AS max_time_to_hire_days
FROM applications a
JOIN jobs j ON a.job_id = j.job_id
WHERE a.hired_flag = 1
GROUP BY j.job_id, j.job_title
ORDER BY avg_time_to_hire_days;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_4092\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,job_id,job_title,hired_count,avg_time_to_hire_days,min_time_to_hire_days,max_time_to_hire_days
0,2,カスタマーサポート,1,28.00,28,28


### 04. 媒体別採用パフォーマンス

**ビジネス課題:** 求人媒体ごとの応募数・採用数・転換率を比較する。

**使用テクニック:** `LEFT JOIN`, `CASE`, `NULLIF`

In [5]:
sql = '''
-- 媒体ごとに応募総数と採用数を集計し、転換率を算出
SELECT
    s.source_id,
    s.source_name,
    s.source_type,
    COUNT(*) AS applications,
    SUM(CASE WHEN a.hired_flag = 1 THEN 1 ELSE 0 END) AS hires,
    ROUND(SUM(CASE WHEN a.hired_flag = 1 THEN 1 ELSE 0 END) /
          NULLIF(COUNT(*), 0) * 100, 2) AS app_to_hire_pct
FROM applications a
LEFT JOIN sources s ON a.source_id = s.source_id
GROUP BY s.source_id, s.source_name, s.source_type
ORDER BY app_to_hire_pct DESC;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_4092\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,source_id,source_name,source_type,applications,hires,app_to_hire_pct
0,2,Referral,referral,6,1.00,16.67
1,5,LinkedIn,job_board,5,0.00,0.00
2,3,Company Website,direct,5,0.00,0.00
3,4,Agency X,agency,7,0.00,0.00
4,1,JobBoard A,job_board,2,0.00,0.00


### 05. 現在のパイプライン（最新ステージ別件数）

**ビジネス課題:** 各候補者の現在のステージを特定し、滞留状況を可視化する。

**使用テクニック:** `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ... DESC)`

In [6]:
sql = '''
-- Step 1: 各応募の最新ステージを ROW_NUMBER() で特定
WITH latest_stage AS (
    SELECT
        s.application_id,
        s.stage_name,
        s.stage_entered_at,
        ROW_NUMBER() OVER (
            PARTITION BY s.application_id
            ORDER BY s.stage_entered_at DESC
        ) AS rn
    FROM application_stages s
)
-- Step 2: 最新ステージ（rn = 1）のみ抽出し、求人×ステージ別に件数を集計
SELECT
    j.job_id,
    j.job_title,
    ls.stage_name,
    COUNT(*) AS count_in_stage
FROM latest_stage ls
JOIN applications a ON ls.application_id = a.application_id
JOIN jobs j        ON a.job_id = j.job_id
WHERE ls.rn = 1
GROUP BY j.job_id, j.job_title, ls.stage_name
ORDER BY j.job_id, ls.stage_name;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_4092\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,job_id,job_title,stage_name,count_in_stage
0,1,内勤営業,applied,1
1,1,内勤営業,offer,1
2,1,内勤営業,rejected,1
3,2,カスタマーサポート,screening,2
4,2,カスタマーサポート,interview,1
5,2,カスタマーサポート,hired,1
6,3,システムエンジニア,interview,2
7,4,データアナリスト,screening,2
8,4,データアナリスト,interview,1
9,4,データアナリスト,offer,1


---

In [7]:
con.close()
print('接続を閉じました。')

接続を閉じました。
